# Zhang (2005) — replicação das tabelas do modelo

Este notebook replica as tabelas **model-only** do projeto de replicação de Zhang (2005), usando a parametrização com **5.000 firmas** e **900 meses**. A sequência é:

1. calibração;
2. solução do equilíbrio aproximado;
3. simulação do painel de firmas;
4. construção das carteiras;
5. momentos e tabelas do modelo;
6. exportação das tabelas.

> Observação: a execução completa com grids benchmark pode ser pesada. Para testes rápidos, reduza `N_FIRMS`, `T_MONTHS`, `NKP`, `KS_ITERATIONS` e `VFI_ITERATIONS`.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

# Ajuste automático do caminho do projeto.
# O notebook pode estar na raiz do projeto ou dentro de uma pasta notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "zhang2005").exists() and (PROJECT_ROOT.parent / "zhang2005").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from zhang2005.calibration import calibrate, simulate_ar1
from zhang2005.equilibrium import construct_equilibrium
from zhang2005.interpolation import interp1_extrap
from zhang2005.portfolios import value_premium
from zhang2005.simulation import simulate_panel
from zhang2005.model_tables import (
    parameter_rows,
    aggregate_moment_rows,
    sort_by_book_to_market,
    portfolio_summary_rows,
    predictive_regression_rows,
)

print("Project root:", PROJECT_ROOT)


## 1. Configuração da replicação

A configuração abaixo usa a escala informada: **5.000 firmas** e **900 meses**. O parâmetro `NKP=5000` usa o grid de candidatos para \(k'\) mais próximo do benchmark do autor. Para depuração, reduza para `NKP=501` ou `NKP=1001`.


In [ ]:
N_FIRMS = 5_000
T_MONTHS = 900
NKP = 5_000

# Sugestão: 120 meses = 10 anos de burn-in para a regressão da lei agregada.
CUTOFF = 120

# Carteiras usam defasagem de 60 meses, como em ValPrem.m.
LAG = 60

# Iterações externas do tipo Krusell-Smith.
KS_ITERATIONS = 20

# Iterações máximas da VFI em cada passo KS.
VFI_ITERATIONS = 10_000

SEED = 20250622
MATLAB_COMPAT = True

OUT_DIR = PROJECT_ROOT / "outputs" / "tables_5000x900"
OUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)


## 2. Calibração do modelo

A função `calibrate()` constrói os parâmetros estruturais, os processos de Markov para produtividade agregada e idiossincrática, o grid de capital, o grid de \(k'\), o grid de preço \(h\), a taxa livre de risco por estado agregado e os tamanhos de simulação.


In [ ]:
cal = calibrate(N=N_FIRMS, Ts=T_MONTHS, nkp=NKP)

table1_parameters = pd.DataFrame(parameter_rows(cal))
table1_parameters


In [ ]:
print("Resumo dos grids")
print(f"N firmas: {cal.N}")
print(f"T meses: {cal.Ts}")
print(f"n_k: {cal.k.size}")
print(f"n_kp: {cal.kp.size}")
print(f"n_h: {cal.h.size}")
print(f"n_x: {cal.nx}")
print(f"n_z: {cal.nz}")
print(f"k min/max: {cal.k.min():.4f} / {cal.k.max():.4f}")
print(f"xbar: {cal.xbar:.8f}")


## 3. Solução do equilíbrio aproximado

Esta célula resolve o loop de equilíbrio aproximado. A lógica é:

\[
\text{coeficientes conjecturados}
\rightarrow
\text{VFI da firma}
\rightarrow
\text{simulação}
\rightarrow
\text{OLS da lei agregada}
\rightarrow
\text{novos coeficientes}.
\]

A lei agregada estimada é aproximadamente:

\[
h_{t+1}
=
a_0+a_1h_t+a_2(x_t-\bar{x})+a_3\sigma_{k,t}+u_{t+1}.
\]


In [ ]:
initial_coefficients = (0.0, 1.0, 0.0, 0.0)

eq = construct_equilibrium(
    cal,
    initial_coefficients=initial_coefficients,
    cutoff=CUTOFF,
    coefficient_tol=1e-2,
    max_iterations=KS_ITERATIONS,
    vfi_options={
        "max_iter": VFI_ITERATIONS,
        "tol_v": 1e-3,
        "tol_k": 1e-8,
        "progress_every": 50,
        "require_convergence": False,
    },
    require_convergence=False,
    rng=rng,
)

eq_summary = pd.DataFrame({
    "object": [
        "KS iterations",
        "Coefficient error",
        "R2",
        "sighat",
        "VFI iterations final",
        "VFI errV final",
        "VFI errK final",
    ],
    "value": [
        eq.iterations,
        eq.error,
        eq.r2,
        eq.sighat,
        eq.solution.iterations,
        eq.solution.errV,
        eq.solution.errK,
    ]
})

coef_table = pd.DataFrame({
    "coefficient": ["alp1", "alp2", "alp3", "alp4"],
    "estimate": eq.coefficients,
    "tstat": eq.tstats,
})

display(eq_summary)
display(coef_table)


## 4. Simulação do painel de firmas

Com a política ótima \(k'=g(k,h,x,z)\) e a função valor \(V(k,h,x,z)\), simulamos 5.000 firmas por 900 meses. Esse painel é a base para preços, book values, dividendos, retornos, investimento, HML, SMB e regressões.


In [ ]:
sx, _ = simulate_ar1(cal.xbar, cal.xbar, cal.rhox, cal.stdx, cal.Ts, rng)

panel = simulate_panel(
    cal,
    eq.simulation.kd,
    eq.simulation.zd,
    sx,
    eq.solution.optK,
    eq.solution.V,
    rng,
)

# Alinhamento temporal: Rf, Rm e srf têm um mês a menos do que Pf/Bf/Df.
Pf = np.maximum(panel.Pf[:, :-1], 1e-8)
Bf = panel.Bf[:, :-1]
Df = panel.Df[:, :-1]
In = panel.In[:, :-1]
Rf = panel.Rf
Rm = panel.Rm
srf = interp1_extrap(cal.x, cal.rf, sx).reshape(-1)[:-1]

panel_diagnostics = pd.DataFrame({
    "object": [
        "Pf shape",
        "Bf shape",
        "Rf shape",
        "Rm shape",
        "Bf min",
        "Bf mean",
        "Bf max",
        "Mean cross-sectional std(Bf)",
        "I/Y",
        "I/K",
        "Adjustment cost / investment",
        "D/Y",
        "Fixed cost / Y",
    ],
    "value": [
        str(Pf.shape),
        str(Bf.shape),
        str(Rf.shape),
        str(Rm.shape),
        Bf.min(),
        Bf.mean(),
        Bf.max(),
        np.mean(np.std(Bf, axis=0, ddof=1)),
        panel.iyr,
        panel.ikr,
        panel.theta,
        panel.dyr,
        panel.fyr,
    ],
})

panel_diagnostics


## 5. Carteiras Fama-French 2x3: Mkt-Rf, SMB, HML, SL, SM, SH, BL, BM, BH

Esta é a tabela análoga à saída de `ValPrem.m`. As linhas são média percentual mensal, desvio-padrão percentual mensal e \(t\)-stat. As colunas são:

\[
Mkt-Rf,\ SMB,\ HML,\ SL,\ SM,\ SH,\ BL,\ BM,\ BH.
\]


In [ ]:
vp = value_premium(
    Pf,
    Rf,
    Bf,
    Rm,
    srf,
    lag=LAG,
    matlab_compat=MATLAB_COMPAT,
)

ff_columns = ["Mkt-Rf", "SMB", "HML", "SL", "SM", "SH", "BL", "BM", "BH"]
ff_rows = ["Mean (%)", "Std (%)", "t-stat"]

table_ff_2x3 = pd.DataFrame(vp.table, index=ff_rows, columns=ff_columns)
table_ff_2x3


## 6. Tabela de momentos agregados do modelo

Esta tabela resume momentos agregados simulados: prêmio de mercado, volatilidade, book-to-market da indústria, dispersão cross-section de B/M, retornos médios das firmas, HML e SMB.


In [ ]:
data = {
    "Pf": Pf,
    "Bf": Bf,
    "Df": Df,
    "In": In,
    "Rf": Rf,
    "Rm": Rm,
    "srf": srf,
    "sx": sx[:Pf.shape[1]],
    "SMB": vp.SMB,
    "HML": vp.HML,
    "coefficients": eq.coefficients,
    "eq_sx": eq.sx,
    "eq_simh": eq.simulation.simh,
    "eq_sigk": eq.simulation.sigk,
    "eq_sigz": eq.simulation.sigz,
    "cutoff": np.asarray(CUTOFF),
}

table2_moments = pd.DataFrame(aggregate_moment_rows(data))
table2_moments


## 7. Carteiras ordenadas por book-to-market

Aqui construímos carteiras value-weighted ordenadas por B/M. Por default usamos 10 carteiras, de `BM1` até `BM10`, mais o spread `HML10 = BM10 - BM1`.

A tabela reporta retorno médio anualizado, volatilidade anualizada, beta de mercado, \(t\)-stat do beta, \(R^2\) CAPM e book-to-market médio.


In [ ]:
bm_sort = sort_by_book_to_market(Pf, Rf, Bf, n_portfolios=10)

table3_bm_portfolios = pd.DataFrame(
    portfolio_summary_rows(bm_sort, Rm, srf)
)

table3_bm_portfolios


## 8. Regressões preditivas do modelo

As regressões abaixo analisam a relação entre HML, value spread, produtividade agregada e book-to-market da indústria. Essas tabelas são as versões **model-only** das regressões preditivas.


In [ ]:
table5_6_predictive = pd.DataFrame(
    predictive_regression_rows(data, bm_sort, cal.xbar)
)

table5_6_predictive


## 9. Salvar resultados

Esta célula salva:

- tabelas em CSV;
- tabelas em Excel;
- arrays simulados em `.npz`;
- resumo da execução em `.json`.

O Excel terá uma aba para cada tabela.


In [ ]:
# CSVs
table1_parameters.to_csv(OUT_DIR / "table1_model_parameters.csv", index=False)
table2_moments.to_csv(OUT_DIR / "table2_model_moments.csv", index=False)
table_ff_2x3.to_csv(OUT_DIR / "table_ff_2x3_value_premium.csv")
table3_bm_portfolios.to_csv(OUT_DIR / "table3_bm_portfolios_model.csv", index=False)
table5_6_predictive.to_csv(OUT_DIR / "table5_6_predictive_regressions_model.csv", index=False)
eq_summary.to_csv(OUT_DIR / "equilibrium_summary.csv", index=False)
coef_table.to_csv(OUT_DIR / "equilibrium_coefficients.csv", index=False)

# Excel
excel_path = OUT_DIR / "zhang2005_model_tables_5000x900.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    table1_parameters.to_excel(writer, sheet_name="Table1_parameters", index=False)
    table2_moments.to_excel(writer, sheet_name="Table2_moments", index=False)
    table_ff_2x3.to_excel(writer, sheet_name="FF_2x3")
    table3_bm_portfolios.to_excel(writer, sheet_name="Table3_BM_portfolios", index=False)
    table5_6_predictive.to_excel(writer, sheet_name="Tables5_6_regressions", index=False)
    eq_summary.to_excel(writer, sheet_name="Equilibrium_summary", index=False)
    coef_table.to_excel(writer, sheet_name="Equilibrium_coef", index=False)

# NPZ com objetos principais para figuras e diagnósticos
npz_path = OUT_DIR / "zhang2005_model_arrays_5000x900.npz"
np.savez_compressed(
    npz_path,
    coefficients=eq.coefficients,
    tstats=eq.tstats,
    residuals=eq.residuals,
    optK=eq.solution.optK,
    V=eq.solution.V,
    Pf=Pf,
    Bf=Bf,
    Df=Df,
    In=In,
    Rf=Rf,
    Rm=Rm,
    srf=srf,
    sx=sx,
    eq_sx=eq.sx,
    eq_simh=eq.simulation.simh,
    eq_sigk=eq.simulation.sigk,
    eq_sigz=eq.simulation.sigz,
    cutoff=np.asarray(CUTOFF),
    panel_simh=panel.simh,
    SMB=vp.SMB,
    HML=vp.HML,
    table_ff_2x3=vp.table,
    bm_returns=bm_sort.returns,
    bm_book_to_market=bm_sort.book_to_market,
    bm_value_spread=bm_sort.value_spread,
)

summary = {
    "seed": SEED,
    "settings": {
        "n_firms": N_FIRMS,
        "periods": T_MONTHS,
        "nkp": NKP,
        "cutoff": CUTOFF,
        "lag": LAG,
        "ks_iterations": KS_ITERATIONS,
        "vfi_iterations": VFI_ITERATIONS,
        "matlab_compat": MATLAB_COMPAT,
    },
    "equilibrium": {
        "coefficients": eq.coefficients.tolist(),
        "r2": float(eq.r2),
        "sighat": float(eq.sighat),
        "error": float(eq.error),
        "iterations": int(eq.iterations),
        "vfi_iterations_final": int(eq.solution.iterations),
        "vfi_errV_final": float(eq.solution.errV),
        "vfi_errK_final": float(eq.solution.errK),
    },
    "aggregate_ratios": {
        "iyr": panel.iyr,
        "ikr": panel.ikr,
        "theta": panel.theta,
        "dyr": panel.dyr,
        "fyr": panel.fyr,
    },
}

json_path = OUT_DIR / "zhang2005_model_summary_5000x900.json"
json_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

print("Arquivos salvos em:", OUT_DIR)
print("Excel:", excel_path)
print("Arrays:", npz_path)
print("Resumo:", json_path)


## 10. Alternativa via scripts

Depois de validar o notebook, a mesma rotina pode ser rodada pelo terminal em duas etapas:

```bash
python scripts/run_mini_replication.py \
  --n-firms 5000 \
  --periods 900 \
  --nkp 5000 \
  --ks-iterations 20 \
  --vfi-iterations 10000 \
  --cutoff 120 \
  --lag 60 \
  --matlab-compat \
  --output-dir outputs/replication_5000x900
```

Depois:

```bash
python scripts/run_model_tables.py \
  --replication-npz outputs/replication_5000x900/mini_replication_results.npz \
  --summary-json outputs/replication_5000x900/mini_replication_summary.json \
  --output-dir outputs/tables/model_5000x900
```
